# Lab 3.2 &mdash; Perception &mdash; Raw Output Is Not an Observation

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Turn an opaque upstream record into a typed observation with a schema
- Distinguish the four kinds of &ldquo;nothing&rdquo; a tool can return
- Stamp in what the agent cannot see &mdash; time, authority, provenance
- Watch the same model answer well and badly on the same facts

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 3.1.** Memory decides what the agent still knows. Perception decides
> what it ever knew in the first place.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 3 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

An agent does not see the world. It sees whatever your tool returned, rendered as text. Handing
a model a raw upstream payload and hoping is the commonest reason a competent agent gives an
incompetent answer.

**Perception** is the step between: decode the record, resolve the codes, add what the model
cannot know, and say plainly what is missing. `with_structured_output` gives you somewhere to put
the result that is checkable.

## Section 1 &mdash; Decode the opaque record

This is what the upstream ledger actually returns. Every field is a code, an epoch, or a flag.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

RAW = {
    "id": "PMT-1005",
    "amt": 75000000,                  # minor units
    "cur": 840,                       # ISO 4217 numeric
    "st": 3,                          # 1 settled, 2 failed, 3 held
    "rc": "SR",                       # abbreviated reason code
    "vd": 1788393600,                 # value date, epoch seconds (2026-09-03)
    "cp": "NORTHWIND",
}

CURRENCIES = {840: "USD", 978: "EUR", 826: "GBP"}
STATUSES = {1: "settled", 2: "failed", 3: "held"}
REASONS = {"IF": "INSUFFICIENT_FUNDS", "LB": "LIMIT_BREACH",
           "II": "INVALID_IBAN", "SR": "SANCTIONS_REVIEW"}


class Observation(BaseModel):
    """What the agent is actually told about one payment."""
    ref: str
    amount: float = Field(description="In major units, not minor")
    currency: str = Field(description="Three-letter code, e.g. USD")
    status: Literal["settled", "failed", "held"]
    reason_code: Optional[str] = Field(description="Expanded reason code, or None")
    value_date: str = Field(description="ISO date, e.g. 2026-09-03")


def perceive(raw: dict) -> Observation:
    """Turn the upstream record into something a model can reason about."""
    return Observation(
        ref=raw["id"],
        amount=raw["amt"] / 100,      # the ledger speaks in cents; the model does not
        currency=CURRENCIES.get(raw["cur"], f"UNKNOWN({raw['cur']})"),
        status=STATUSES[raw["st"]],
        reason_code=REASONS.get(raw["rc"]),
        value_date=time.strftime("%Y-%m-%d", time.gmtime(raw["vd"])),
    )

In [ ]:
# --- Self-check: Section 1   (pure decoding -- no model call)
check("the amount is in major units",
      lambda: perceive(RAW).amount == 750000.0,
      "75000000 minor units is USD 750,000 -- a model told 75000000 will reason about the wrong number")
check("the currency code is resolved",  lambda: perceive(RAW).currency == "USD")
check("the status is resolved",         lambda: perceive(RAW).status == "held")
check("the reason code is expanded",    lambda: perceive(RAW).reason_code == "SANCTIONS_REVIEW",
      '"SR" means nothing to a model; SANCTIONS_REVIEW appears in the policy catalogue')
check("the epoch is a readable date",   lambda: perceive(RAW).value_date.startswith("2026-09"))
def _rejects_bad_status():
    try:
        Observation(ref="X", amount=1.0, currency="USD", status="pending",
                    reason_code=None, value_date="2026-09-03")
        return False
    except Exception:
        return True

check("an invalid status is rejected by the schema",
      lambda: _rejects_bad_status(),
      "Literal[...] means a decoding bug fails here, not three steps later in a policy lookup")

## Section 2 &mdash; The four kinds of nothing

An empty result is not one thing. "No such payment", "no permission to see it", "the upstream is
down" and "it exists and has no reason code" all arrive as something falsy, and they require four
different responses. Collapsing them is how an agent confidently reports that a payment does not
exist when it simply could not read it.

In [ ]:
def describe_empty(result: dict) -> str:
    """Say which kind of nothing this is."""
    if result.get("error") == "not_found":
        return "no such payment exists"
    if result.get("error") == "forbidden":
        return "not permitted to read this payment; existence unknown"
    if result.get("error") in ("timeout", "unavailable"):
        return "could not read the ledger; state unknown"
    if result.get("record") is not None and not result["record"].get("rc"):
        return "the payment exists and has no reason code"
    return "unrecognised result shape"

In [ ]:
# --- Self-check: Section 2
check("a missing record says so",
      lambda: "no such payment" in describe_empty({"error": "not_found"}))
check("a permission failure does NOT claim the payment is missing",
      lambda: "no such payment" not in describe_empty({"error": "forbidden"}),
      "this is the dangerous one: 'I cannot see it' is not 'it is not there'")
check("a permission failure says existence is unknown",
      lambda: "unknown" in describe_empty({"error": "forbidden"}).lower())
check("an outage says state unknown",
      lambda: "unknown" in describe_empty({"error": "timeout"}).lower())
check("a real record with no reason code is not an error",
      lambda: "exists" in describe_empty({"record": {"id": "PMT-1001"}}))
check("all four kinds give different answers",
      lambda: len({describe_empty({"error": e}) for e in
                   ("not_found", "forbidden", "timeout")}) == 3)

## Section 3 &mdash; Stamp in what the agent cannot see

The model has no clock, no idea who is asking, and no memory of where a fact came from. If those
matter to the decision &mdash; and here they do &mdash; they have to be in the observation.

In [ ]:
NOW = 1788393600 + 7200               # pretend "now" is two hours after the value date

class Context(BaseModel):
    """Everything true of the situation rather than of the payment."""
    observed_at: str = Field(description="ISO timestamp when this was read")
    hours_since_value_date: float
    source: str = Field(description="Which system this came from, for provenance")
    caller_may_release: bool = Field(description="Whether the human asking has release authority")


def contextualise(obs: Observation, raw: dict, caller_role: str) -> Context:
    return Context(
        observed_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(NOW)),
        hours_since_value_date=round((NOW - raw["vd"]) / 3600, 1),
        source="ledger-core",
        # Operations may release ordinary holds but never one policy reserves for a human
        # decision -- a sanctions review is Compliance's call whoever is asking.
        caller_may_release=(caller_role == "compliance"
                            or (caller_role == "operations"
                                and obs.reason_code not in NEEDS_HUMAN)),
    )

In [ ]:
# --- Self-check: Section 3
# built lazily: perceive() may still contain a blank, and a module-level call would
# crash this cell instead of reporting [TODO]
_obs   = lambda: perceive(RAW)                       # PMT-1005, SANCTIONS_REVIEW
_clean = lambda: perceive({**RAW, "rc": "IF"})       # same payment, an ordinary failure

check("the observation is stamped with a time",
      lambda: contextualise(_obs(), RAW, "operations").observed_at.startswith("2026-"))
check("elapsed time is computed, not left to the model",
      lambda: contextualise(_obs(), RAW, "operations").hours_since_value_date == 2.0,
      "a model asked to subtract two epochs will sometimes get it wrong; do it in Python")
check("operations may NOT release a sanctions hold",
      lambda: contextualise(_obs(), RAW, "operations").caller_may_release is False,
      "policy reserves this decision for Compliance -- authority is context, not preference")
check("operations MAY release an ordinary failure",
      lambda: contextualise(_clean(), RAW, "operations").caller_may_release is True)
check("compliance may release a sanctions hold",
      lambda: contextualise(_obs(), RAW, "compliance").caller_may_release is True)
check("an unknown role gets no authority",
      lambda: contextualise(_clean(), RAW, "intern").caller_may_release is False,
      "default deny -- an unrecognised role is not a permitted one")

## Run it for real

The same model, the same underlying payment, asked the same question. Once from the raw record,
once from the observation and its context.

In [ ]:
if llm_ready():
    def _compare():
        question = ("Who must action this payment, and may the operations desk release it? "
                    "Answer in two lines.")
        obs = perceive(RAW)
        ctx = contextualise(obs, RAW, "operations")
        policy = POLICY.get(obs.reason_code, "no policy applies")

        print("=== given the RAW record ===")
        print(ask(f"RECORD: {json.dumps(RAW)}\n\n{question}")[:400])

        print("\n=== given the OBSERVATION and its context ===")
        print(ask(f"OBSERVATION: {obs.model_dump_json()}\n"
                  f"CONTEXT: {ctx.model_dump_json()}\n"
                  f"POLICY: {policy}\n\n{question}")[:400])
    guard(_compare)

### Read it

The raw answer is the interesting one. The model has to guess that `st: 3` means held, that `rc`
is a reason code at all, that `amt` is in cents, and that `cur: 840` is dollars. Most of the time
it will guess several of those correctly and state the rest with complete confidence &mdash; which is
exactly the failure you cannot detect downstream, because nothing looks wrong.

The second answer is not smarter. It is the same model, given the same facts, in a form it does
not have to decode. Note in particular that `caller_may_release` was decided by **your code**,
against the policy, before the model saw anything. Asking a model to work out who is authorised
is asking it to make a control decision; computing it in `contextualise` and telling it the answer
is not.

That is the general rule this lab is for: **anything you can determine in Python, determine in
Python.** Leave the model the part that actually needs judgement.

In [ ]:
score()

## Your turn

1. Add `cur: 392` (JPY) to `CURRENCIES` &mdash; but the yen has no minor units, so `amt` is already
   in major units. Where does that belong: in `perceive`, in the schema, or in the tool that
   produced the record? Defend your answer.
2. Feed `describe_empty` into the run-it-for-real cell: ask the model what to do when the ledger
   returns `{"error": "forbidden"}`, once with the raw error and once with your description. Watch
   how often the raw version concludes the payment does not exist.
3. `Observation` has no field for what is **missing**. Add `unknown: list[str]` and populate it
   when a code fails to resolve. An agent that can say "I do not know the currency" is worth more
   than one that quietly says `UNKNOWN(392)`.